<a href="https://colab.research.google.com/github/fdx-hw/cosc-650/blob/main/Week_3_discussion_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 Discussion (Prompt Failure & Structural Fix): "Is This Game Worth Buying?"


## The task

**Read a pile of critic coverage about a specific game and produce a purchase
verdict** (buy / wait / skip, with reasons) — the thing I'm actually trying to
decide for myself before September 15.

The test corpus below is real, current coverage, gathered today:

- Genuine **Marvel's Wolverine (2026)** preview quotes from Eurogamer, GameSpot,
  GamesRadar+, IGN, Skill Up, and Tech4Gamers (all published mid-August 2026,
  ahead of the review embargo — these are hands-on previews, *not* final scored
  reviews).
- Two **distractor sources that also contain the word "Wolverine" but are about
  something else entirely**: a Metacritic entry for *X-Men Origins: Wolverine*
  (a 2009 movie tie-in game, unrelated developer, Metascore 75) and a Rotten
  Tomatoes figure for the *Deadpool & Wolverine* (2024) film. If you searched
  "Wolverine reviews" today and pasted the top results into a prompt without
  reading carefully, both of these would plausibly end up in your clipboard.
- One **forum speculation post** (a ResetEra score-prediction thread comment) —
  not a review at all, just a guess.

This mix is deliberate: it's exactly what naive copy-paste-from-search research
actually looks like, and it sets up a specific, real failure mode.


In [ ]:
!pip install -q anthropic

import json
import re
from google.colab import userdata
import anthropic

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
MODEL = "claude-sonnet-4-5"
print(anthropic.__version__)


1.4.0


In [ ]:
SOURCES = [
    # --- Genuine Marvel's Wolverine (2026) preview coverage ---
    {"outlet": "Eurogamer", "subject": "Marvel's Wolverine", "year": 2026, "medium": "game",
     "review_type": "preview",
     "text": "Combat is ferocious, destructive and cinematic -- but a little predictable."},
    {"outlet": "GameSpot", "subject": "Marvel's Wolverine", "year": 2026, "medium": "game",
     "review_type": "preview",
     "text": "There's real satisfaction to the combat, and the Rage mechanic ties nicely into Logan's emotional state."},
    {"outlet": "GamesRadar+", "subject": "Marvel's Wolverine", "year": 2026, "medium": "game",
     "review_type": "preview",
     "text": "This preview crushed my fears of a post-Spider-Man slump for Insomniac."},
    {"outlet": "Skill Up", "subject": "Marvel's Wolverine", "year": 2026, "medium": "game",
     "review_type": "preview",
     "text": "The story is wonderfully told, with standout performances from Troy Baker and Liam McIntyre."},
    {"outlet": "IGN", "subject": "Marvel's Wolverine", "year": 2026, "medium": "game",
     "review_type": "preview",
     "text": "My biggest concern after this preview: Wolverine mostly just angrily stabs things, and I'm not yet sure the combat has enough variety to sustain a full campaign."},
    {"outlet": "Tech4Gamers", "subject": "Marvel's Wolverine", "year": 2026, "medium": "game",
     "review_type": "preview",
     "text": "A decently solid action game, but nothing revolutionary -- targeting struggles in crowded encounters, and the story beats are fairly standard, arguably a step down from the Spider-Man titles."},

    # --- Distractors: same keyword ("Wolverine"), completely different subject ---
    {"outlet": "Metacritic (critic aggregate)", "subject": "X-Men Origins: Wolverine", "year": 2009, "medium": "game",
     "review_type": "final_review",
     "text": "Metascore 75 (Xbox 360). One critic noted: 'this is another case of a game falling victim to the tie-in; cutting corners to meet a short deadline.'"},
    {"outlet": "Rotten Tomatoes (critic aggregate)", "subject": "Deadpool & Wolverine", "year": 2024, "medium": "film",
     "review_type": "final_review",
     "text": "Critics score: 81% on Rotten Tomatoes."},
    {"outlet": "ResetEra forum user", "subject": "Marvel's Wolverine", "year": 2026, "medium": "game",
     "review_type": "forum_speculation",
     "text": "Calling it now, this is getting an 89 on Metacritic, Insomniac always lands in the high 80s to low 90s."},
]

def sources_to_plain_blob(sources):
    """What a naive copy-paste-from-search-results workflow actually produces:
    just the review text, run together, no labels."""
    return "\n\n".join(s["text"] for s in sources)

SHORT_CLEAN_SOURCES = SOURCES[:3]          # only unambiguous, genuine, positive previews
LONG_MESSY_SOURCES = SOURCES               # everything, including both distractors + forum noise


## Baseline prompt

Simple and exactly how a first draft of this actually looks: no source labels,
no schema, no instruction about what to do with a source that turns out not to
be about the right thing.


In [ ]:
BASELINE_PROMPT = """I'm trying to decide whether to buy Marvel's Wolverine. Here
is a collection of review snippets I found searching around online. Based on
these, should I buy it? Give me your recommendation and why.

Reviews:
{reviews}"""

def run_baseline(sources, temperature=0.7):
    resp = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        extra_body={"temperature": temperature},
        messages=[{"role": "user", "content": BASELINE_PROMPT.format(reviews=sources_to_plain_blob(sources))}],
    )
    return resp.content[0].text


### Prediction

On `SHORT_CLEAN_SOURCES` (three genuine, unambiguous, positive previews) I expect
the baseline prompt to do fine — it should recommend buying, and *might* even
independently think to mention these are pre-release previews rather than final
reviews, since nothing here labels them that way either.

On `LONG_MESSY_SOURCES` I expect a specific failure: since nothing distinguishes
the *X-Men Origins: Wolverine* (2009) or *Deadpool & Wolverine* (2024 film)
entries from the real ones — they share the keyword and get pasted into the same
undifferentiated blob — I expect the model to fold their scores/quotes into its
reasoning about the 2026 game. Concretely, I'd expect the response to cite the
X-Men Origins Metascore of 75 or the 81% Rotten Tomatoes figure as if either were
evidence about *this* game, and/or to treat the ResetEra forum guess ("89 on
Metacritic") as if it were an actual published score. I'd also expect it to drop
the fact that none of the genuine sources are final reviews at all — the embargo
hasn't lifted yet.

This is **context contamination via entity confusion**: the failure isn't
irrelevant chatter bleeding in (like a meeting transcript's small talk), it's
same-keyword-different-entity material being silently treated as on-topic.


In [ ]:
def mentions_contamination(text):
    """Cheap detector: did the response surface numbers/phrases that only exist
    in the distractor sources, as if they applied to the 2026 game?"""
    hits = []
    if "75" in text:
        hits.append("X-Men Origins: Wolverine Metascore (75)")
    if "81%" in text or "81 %" in text or "81 percent" in text.lower():
        hits.append("Deadpool & Wolverine Rotten Tomatoes score (81%)")
    if "89" in text:
        hits.append("ResetEra forum speculation ('89 on Metacritic') treated as real")
    return hits

def mentions_preview_caveat(text):
    keywords = ["preview", "not yet released", "embargo", "pre-release", "hasn\'t launched", "hasn\'t come out"]
    return any(k in text.lower() for k in keywords)

print("=== SHORT_CLEAN_SOURCES, baseline prompt ===\n")
short_out = run_baseline(SHORT_CLEAN_SOURCES)
print(short_out)
print("\n[contamination hits:", mentions_contamination(short_out), "]")
print("[flags preview/pre-release status:", mentions_preview_caveat(short_out), "]")


=== SHORT_CLEAN_SOURCES, baseline prompt ===

# Recommendation: **Wait for more reviews**

Here's why I'd hold off for now:

## What we know (limited info):
- **Positives**: Combat seems solid and viscerally satisfying, with good thematic integration (Rage mechanic fitting Wolverine's character)
- **Concerns**: Combat may become repetitive ("predictable")
- **Context issue**: One snippet mentions this is a "preview," not a full game review

## Key problems with making a decision now:

1. **Incomplete picture** - These snippets don't cover crucial aspects like:
   - Story quality
   - Game length/value
   - Variety beyond combat
   - Technical performance
   - Overall pacing

2. **Preview vs. Review** - At least one quote is from a preview, meaning reviewers haven't seen the full game

3. **Mixed signals** - "Ferocious" combat but "predictable" suggests it might get repetitive over a full playthrough

## My advice:
**Wait for full reviews** from trusted sources that cover the complete e

In [ ]:
print("=== LONG_MESSY_SOURCES, baseline prompt, 5 trials @ temperature=0.7 ===\n")
baseline_results = []
for i in range(5):
    out = run_baseline(LONG_MESSY_SOURCES, temperature=0.7)
    hits = mentions_contamination(out)
    flags_preview = mentions_preview_caveat(out)
    baseline_results.append({"trial": i, "raw": out, "contamination": hits, "flags_preview": flags_preview})
    print(f"--- Trial {i} ---")
    print(out)
    print(f"[contamination hits: {hits} | flags preview status: {flags_preview}]\n")


=== LONG_MESSY_SOURCES, baseline prompt, 5 trials @ temperature=0.7 ===

--- Trial 0 ---
I need to point out some **important confusion** in these reviews before giving you a recommendation:

## The Problem
These reviews appear to be mixing up **two completely different games**:

1. **Marvel's Wolverine (upcoming)** - Insomniac's game that hasn't been released yet (only previews exist)
2. **X-Men Origins: Wolverine (2009)** - The old movie tie-in game for Xbox 360/PS3 (Metascore 75)

The mentions of Xbox 360, movie tie-ins, and existing Metascores are clearly referring to the 2009 game, while references to "post-Spider-Man" Insomniac and preview impressions refer to the unreleased title.

## My Recommendation: **WAIT**

**Don't buy yet** because:
- **Insomniac's Marvel's Wolverine isn't out yet** - there's no release date announced
- You can't actually purchase it currently
- The preview impressions are mixed (excited about potential, but concerned about combat variety)

**If you meant

### Actual results (baseline)

*Fill this in after running the cells above.*

- Across the 5 trials on `LONG_MESSY_SOURCES`, how many actually pulled in the
  75, the 81%, or the forum-guessed 89 as if they were evidence about the 2026
  game? Quote the exact sentence where it happens, if it does.
- How many trials correctly flagged that all the genuine sources are
  pre-release previews, not final reviews?
- **Failure mode classification:** this is a candidate for **context
  contamination** (wrong-entity material treated as on-topic because of a
  shared keyword) — does the evidence support that, or did something else go
  wrong instead (e.g. the model actually filtered correctly and the failure
  was elsewhere, like the missing preview/final-review distinction, which
  would be closer to an incomplete-instruction problem)?


## Structural fix

The fix here isn't primarily about output formatting (though the schema below
helps) — it's about **giving the model an explicit way to tell sources apart**
instead of relying on it to infer relevance from raw text alone:

1. Each source is now wrapped with **metadata as structured delimiters**
   (`subject`, `year`, `medium`, `review_type`) instead of being flattened into
   one blob — the same "mark data as data" idea as XML-delimiting a transcript,
   just with richer tags.
2. The **system prompt pins the exact target** (title, developer, platform,
   year) and gives an explicit rule: anything whose `subject`/`year`/`medium`
   doesn't match gets excluded from the reasoning, but named in an
   `excluded_sources` field so the exclusion is auditable rather than silent.
3. An explicit **output schema** requires every pro/con to carry a source
   attribution, plus a `coverage_type` field the model must set honestly
   (`previews_only` / `mixed` / `final_reviews`), which directly targets the
   dropped-caveat failure.
4. An assistant-turn **prefill of `{`** to force valid JSON.

I expect this to eliminate the entity-confusion contamination almost entirely,
since the model no longer has to *infer* that the 2009 game and the 2024 movie
are off-topic — it's told outright what "on-topic" means and given a field to
show its exclusions. I'm less sure it'll perfectly catch the forum-speculation
entry, since that one *is* about the right subject/year/medium — it's just not
a review at all — so I've added a `review_type` tag for exactly that reason,
but whether the model actually uses it correctly is worth checking rather than
assuming.


In [ ]:
FIXED_SYSTEM_PROMPT = """You help the user decide whether to buy a specific
game, based on critic and community coverage they provide.

Target game (the ONLY subject you should reason about):
  title: Marvel's Wolverine
  developer: Insomniac Games
  platform: PS5
  release_year: 2026

The user will paste a list of sources, each wrapped like this:
<source outlet="..." subject="..." year="..." medium="..." review_type="preview|final_review|forum_speculation">
  text
</source>

Rules:
- Only use a source in your reasoning if its subject/year/medium exactly match
  the target game above. If a source is about a different game, a movie, or
  anything else that merely shares a name, you must NOT use it as evidence --
  list it in "excluded_sources" instead, with a one-phrase reason.
- Treat review_type="forum_speculation" as opinion/prediction, not a published
  score or verdict -- it can inform "community_sentiment" but must never be
  reported as a critic score or a final review.
- Set "coverage_type" honestly: "previews_only" if every included source is a
  preview, "final_reviews" if every included source is a final review, or
  "mixed" if both are present.

Output a single JSON object with exactly these keys:
  "recommendation": one of "buy", "wait", "skip"
  "confidence": one of "low", "medium", "high"
  "coverage_type": "previews_only" | "mixed" | "final_reviews"
  "key_pros": array of {{"point": string, "source": string}}
  "key_cons": array of {{"point": string, "source": string}}
  "community_sentiment": string or null
  "excluded_sources": array of {{"outlet": string, "reason": string}}

Respond with ONLY the JSON object. No preamble, no closing remarks, no markdown
code fence."""

def sources_to_tagged_blob(sources):
    parts = []
    for s in sources:
        parts.append(
            f'<source outlet="{s["outlet"]}" subject="{s["subject"]}" '
            f'year="{s["year"]}" medium="{s["medium"]}" review_type="{s["review_type"]}">\n'
            f'{s["text"]}\n</source>'
        )
    return "\n\n".join(parts)

def run_fixed(sources, temperature=0.7):
    resp = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        extra_body={"temperature": temperature},
        system=FIXED_SYSTEM_PROMPT,
        messages=[
            {"role": "user", "content": sources_to_tagged_blob(sources)},
            {"role": "assistant", "content": "{"},
        ],
    )
    return "{" + resp.content[0].text


### Prediction (fixed prompt)

I expect `excluded_sources` to correctly list the *X-Men Origins: Wolverine*
and *Deadpool & Wolverine* entries in every trial, since their `subject`/`year`/
`medium` tags fail the match unambiguously. I expect `coverage_type` to read
`"previews_only"` consistently, since every genuine, on-topic source is tagged
`review_type="preview"`. The open question is the forum-speculation entry:
I predict it'll show up in `community_sentiment` rather than being counted as a
critic score, but since that requires the model to actually apply the
`review_type` rule rather than just the subject/year/medium match, I'm running
multiple trials rather than assuming one clean run generalizes.

Running 5 trials at three temperatures (0, 0.7, 1.0) again, to match the
coverage expectation from Week 2 feedback rather than judging off one setting.


In [ ]:
def try_parse_json(text):
    try:
        return json.loads(text.strip()), None
    except json.JSONDecodeError as e:
        return None, str(e)

print("=== LONG_MESSY_SOURCES, FIXED prompt, 5 trials x 3 temperatures ===\n")
fixed_results = []
for temp in (0.0, 0.7, 1.0):
    for i in range(5):
        out = run_fixed(LONG_MESSY_SOURCES, temperature=temp)
        parsed, err = try_parse_json(out)
        excluded = [e.get("outlet") for e in parsed.get("excluded_sources", [])] if parsed else []
        coverage = parsed.get("coverage_type") if parsed else None
        contamination = mentions_contamination(out)
        fixed_results.append({"temp": temp, "trial": i, "raw": out, "parsed": parsed,
                               "error": err, "excluded": excluded, "coverage_type": coverage,
                               "contamination": contamination})
        print(f"--- temp={temp} trial={i} | parsed OK: {parsed is not None} | "
              f"coverage_type={coverage} | excluded={excluded} | contamination={contamination} ---")
        print(out)
        print()


=== LONG_MESSY_SOURCES, FIXED prompt, 5 trials x 3 temperatures ===

--- temp=0.0 trial=0 | parsed OK: True | coverage_type=previews_only | excluded=['Metacritic (critic aggregate)', 'Rotten Tomatoes (critic aggregate)'] | contamination=[] ---
{
  "recommendation": "wait",
  "confidence": "medium",
  "coverage_type": "previews_only",
  "key_pros": [
    {
      "point": "Combat is ferocious, destructive and cinematic with satisfying Rage mechanic tied to Logan's emotional state",
      "source": "Eurogamer, GameSpot"
    },
    {
      "point": "Story is wonderfully told with standout performances from Troy Baker and Liam McIntyre",
      "source": "Skill Up"
    },
    {
      "point": "Preview quality suggests no post-Spider-Man slump for Insomniac",
      "source": "GamesRadar+"
    }
  ],
  "key_cons": [
    {
      "point": "Combat is predictable and may lack variety to sustain a full campaign - Wolverine mostly just angrily stabs things",
      "source": "Eurogamer, IGN"
    },
 

In [24]:
# Side-by-side summary: baseline vs. fixed, on the contamination + preview-caveat checks
n_baseline_contaminated = sum(1 for r in baseline_results if r["contamination"])
n_baseline_flagged_preview = sum(1 for r in baseline_results if r["flags_preview"])
n_fixed_contaminated = sum(1 for r in fixed_results if r["contamination"])
n_fixed_correct_coverage = sum(1 for r in fixed_results if r["coverage_type"] == "previews_only")

def excluded_both_distractors(parsed):
    if not parsed:
        return False
    combined = " ".join(
        f"{e.get('outlet', '')} {e.get('reason', '')}" for e in parsed.get("excluded_sources", [])
    )
    return "X-Men" in combined and ("Deadpool" in combined or "Rotten Tomatoes" in combined)

n_fixed_excluded_both = sum(1 for r in fixed_results if excluded_both_distractors(r["parsed"]))

print(f"Baseline:  {n_baseline_contaminated}/{len(baseline_results)} trials showed cross-entity contamination")
print(f"Baseline:  {n_baseline_flagged_preview}/{len(baseline_results)} trials correctly noted preview-only status")
print(f"Fixed:     {n_fixed_contaminated}/{len(fixed_results)} trials showed cross-entity contamination")
print(f"Fixed:     {n_fixed_correct_coverage}/{len(fixed_results)} trials correctly set coverage_type='previews_only'")
print(f"Fixed:     {n_fixed_excluded_both}/{len(fixed_results)} trials correctly excluded BOTH distractor sources")


Baseline:  5/5 trials showed cross-entity contamination
Baseline:  5/5 trials correctly noted preview-only status
Fixed:     0/15 trials showed cross-entity contamination
Fixed:     15/15 trials correctly set coverage_type='previews_only'
Fixed:     15/15 trials correctly excluded BOTH distractor sources
